# Vector Search in DuckDB

In this exercise, you will explore exact and approximate nearest-neighbor search with DuckDB.

In [1]:
from pathlib import Path
import statistics
import time

import duckdb
import pandas as pd

In [2]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DOCS_CSV = DATA_DIR / "docs_part_*.csv"
QUERY_VECTOR_FILE = DATA_DIR / "query_vector.txt"
GT_GLOBAL_FILE = DATA_DIR / "global_topk_ids.txt"
GT_FILTERED_FILE = DATA_DIR / "filtered_sports_topk_ids.txt"
DIM = 128
K_DEFAULT = 10

In [3]:
QUERY_VECTOR_LITERAL = QUERY_VECTOR_FILE.read_text(encoding="utf-8").strip()

SQL_CREATE_DOCS = f"""
CREATE OR REPLACE TABLE docs AS
SELECT
    id::INTEGER AS id,
    topic::VARCHAR AS topic,
    title::VARCHAR AS title,
    embedding::FLOAT[{DIM}] AS embedding
FROM read_csv_auto('{DOCS_CSV}')
"""

SQL_CREATE_QVEC_MACRO = f"CREATE OR REPLACE MACRO qvec() AS {QUERY_VECTOR_LITERAL}::FLOAT[{DIM}]"

Q1_SQL_TEMPLATE = """
SELECT id, topic, title, array_distance(embedding, qvec()) AS l2_distance
FROM docs
ORDER BY l2_distance ASC
LIMIT {k}
"""

Q2_SQL_TEMPLATE = """
SELECT id, topic, title, array_distance(embedding, qvec()) AS l2_distance
FROM docs
WHERE topic = 'sports'
ORDER BY l2_distance ASC
LIMIT {k}
"""


def load_ids(path):
    return [int(x.strip()) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]


def recall_stats(result_ids, groundtruth_ids, k):
    k_eff = min(k, len(groundtruth_ids))
    if k_eff == 0:
        return 0.0, 0, 0
    gt_topk = set(groundtruth_ids[:k_eff])
    hits = sum(1 for doc_id in result_ids[:k_eff] if doc_id in gt_topk)
    return hits / k_eff, hits, k_eff


def run_query_median(con, sql, runs=5):
    latencies_ms = []
    last_df = None
    for _ in range(runs):
        start = time.perf_counter()
        last_df = con.execute(sql).fetchdf()
        latencies_ms.append((time.perf_counter() - start) * 1000)
    return last_df, statistics.median(latencies_ms), latencies_ms


def print_explain_plan(title, explain_rows):
    print(title)
    for key, value in explain_rows:
        print(f"[{key}]")
        print(value)
        print()

## 0) Environment setup

Run the next cell once before starting the tasks.

In [4]:
k_default = K_DEFAULT

con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=1")
con.execute("PRAGMA memory_limit='1GB'")
con.execute(SQL_CREATE_DOCS)
con.execute(SQL_CREATE_QVEC_MACRO)

vss_ok = False
try:
    con.execute("LOAD vss")
    con.execute("SET hnsw_enable_experimental_persistence = true")
    vss_ok = True
except duckdb.Error:
    try:
        con.execute("INSTALL vss")
        con.execute("LOAD vss")
        con.execute("SET hnsw_enable_experimental_persistence = true")
        vss_ok = True
    except duckdb.Error as exc:
        print("Could not load VSS extension:", exc)

## 1) Global search plan comparison (`EXPLAIN`)

Run the next cell and compare the two plans.

What to look for:
- operator differences between non-indexed and indexed runs,
- whether an index-specific scan operator appears,
- where sorting and limiting happen.


In [5]:
q1_sql = Q1_SQL_TEMPLATE.format(k=k_default)

# --- Exact search plan (no index) ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
plan_q1_no_index = con.execute(f"EXPLAIN {q1_sql}").fetchall()

print_explain_plan("Q1 plan without index", plan_q1_no_index)

# --- HNSW indexed search plan ---

con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
con.execute("SET hnsw_ef_search = 64")
con.execute(
    "CREATE INDEX docs_hnsw_idx ON docs USING HNSW (embedding) "
    "WITH (M = 4, ef_construction = 16)"
)
plan_q1_hnsw = con.execute(f"EXPLAIN {q1_sql}").fetchall()

print_explain_plan("Q1 plan with HNSW", plan_q1_hnsw)

Q1 plan without index
[physical_plan]
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│             #0            │
│__internal_decompress_strin│
│           g(#1)           │
│             #2            │
│             #3            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│ array_distance(memory.main│
│  .docs.embedding, qvec()) │
│             ASC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│             #0            │
│__internal_compress_string_│
│        uhugeint(#1)       │
│             #2            │
│             #3            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│ 

## 2) Global search execution: recall vs latency

Run the next cell to compare exact search and HNSW-indexed search using default settings.

- which mode is faster,
- which mode has higher recall

In [6]:
gt_global = load_ids(GT_GLOBAL_FILE)
rows = []

# --- Exact search (no index) ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
q1_exact_df, q1_exact_latency_ms, _ = run_query_median(con, q1_sql, runs=5)

q1_exact_ids = q1_exact_df["id"].tolist()
q1_exact_recall, q1_exact_hits, q1_exact_k = recall_stats(q1_exact_ids, gt_global, k_default)

rows.append({
    "mode": "no_index",
    "latency_ms": round(q1_exact_latency_ms, 2),
    "recall": round(q1_exact_recall, 3),
    "returned_rows": len(q1_exact_df),
})

# --- HNSW indexed search ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
con.execute("SET hnsw_ef_search = 64")
con.execute(
    "CREATE INDEX docs_hnsw_idx ON docs USING HNSW (embedding) "
    "WITH (M = 4, ef_construction = 16)"
)
q1_hnsw_df, q1_hnsw_latency_ms, _ = run_query_median(con, q1_sql, runs=5)

q1_hnsw_ids = q1_hnsw_df["id"].tolist()
q1_hnsw_recall, q1_hnsw_hits, q1_hnsw_k = recall_stats(q1_hnsw_ids, gt_global, k_default)

rows.append({
    "mode": "hnsw_default",
    "latency_ms": round(q1_hnsw_latency_ms, 2),
    "recall": round(q1_hnsw_recall, 3),
    "returned_rows": len(q1_hnsw_df),
})

pd.DataFrame(rows)

,mode,latency_ms,recall,returned_rows
0,no_index,8.17,1.0,10
1,hnsw_default,1.08,0.8,10


## 3) Parameter sweep: `hnsw_ef_search`

Run indexed global search with `hnsw_ef_search` values `[512, 256, 128, 64, 32, 16]`.

- How does recall change as `hnsw_ef_search` increases?
- How does latency change?
- Which value gives the best balance ?

In [7]:
ef_values = [512, 256, 128, 64, 32, 16]
rows = []

# Build the index once, then only change search-time ef.
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
con.execute(
    "CREATE INDEX docs_hnsw_idx ON docs USING HNSW (embedding) "
    "WITH (M = 4, ef_construction = 16)"
)

for ef in ef_values:
    con.execute(f"SET hnsw_ef_search = {ef}")

    df, latency_ms, _ = run_query_median(con, q1_sql, runs=5)

    result_ids = df["id"].tolist()
    recall, hits, k_eff = recall_stats(result_ids, gt_global, k_default)

    rows.append({
        "hnsw_ef_search": ef,
        "latency_ms": round(latency_ms, 2),
        "recall": round(recall, 3),
    })

pd.DataFrame(rows)

,hnsw_ef_search,latency_ms,recall
0,512,1.23,1.0
1,256,1.06,0.9
2,128,1.00,0.8
3,64,0.94,0.8
4,32,0.95,0.4
5,16,0.95,0.2


## 4) Filtered search plan comparison (`EXPLAIN`)

Now test a filtered query (`topic = 'sports'`).

Run the next cell and inspect both plans.

- In the indexed plan, does filtering happen before candidate generation (pre-filtering) or after it (post-filtering)?

In [8]:
q2_sql = Q2_SQL_TEMPLATE.format(k=k_default)

# --- Exact filtered search plan (no index) ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
plan_q2_no_index = con.execute(f"EXPLAIN {q2_sql}").fetchall()

print_explain_plan("Q2 plan without index", plan_q2_no_index)

# --- HNSW filtered search plan ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
con.execute("SET hnsw_ef_search = 64")
con.execute(
    "CREATE INDEX docs_hnsw_idx ON docs USING HNSW (embedding) "
    "WITH (M = 4, ef_construction = 16)"
)
plan_q2_hnsw = con.execute(f"EXPLAIN {q2_sql}").fetchall()

print_explain_plan("Q2 plan with HNSW", plan_q2_hnsw)

plan_text = "\n".join(str(value) for _, value in plan_q2_hnsw)

Q2 plan without index
[physical_plan]
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│             #0            │
│__internal_decompress_strin│
│           g(#1)           │
│             #2            │
│             #3            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│ array_distance(memory.main│
│  .docs.embedding, qvec()) │
│             ASC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│             #0            │
│__internal_compress_string_│
│        uhugeint(#1)       │
│             #2            │
│             #3            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│ 

## 5) Filtered search execution: 

Run the next cell to compare exact vs indexed execution for filtered search.

Check all three:
- recall,
- latency,
- returned row count.

Verify whether indexed execution returns `k` rows under filtering.

In [9]:
gt_filtered = load_ids(GT_FILTERED_FILE)
rows = []

# --- Exact filtered search (no index) ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
q2_exact_df, q2_exact_latency_ms, _ = run_query_median(con, q2_sql, runs=5)

q2_exact_ids = q2_exact_df["id"].tolist()
q2_exact_recall, q2_exact_hits, q2_exact_k = recall_stats(q2_exact_ids, gt_filtered, k_default)

rows.append({
    "mode": "no_index",
    "latency_ms": round(q2_exact_latency_ms, 2),
    "recall": round(q2_exact_recall, 3),
    "returned_rows": len(q2_exact_df),
    "expected_k": k_default,
})

# --- HNSW filtered search ---
con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
con.execute("SET hnsw_ef_search = 64")
con.execute(
    "CREATE INDEX docs_hnsw_idx ON docs USING HNSW (embedding) "
    "WITH (M = 4, ef_construction = 16)"
)
q2_hnsw_df, q2_hnsw_latency_ms, _ = run_query_median(con, q2_sql, runs=5)

q2_hnsw_ids = q2_hnsw_df["id"].tolist()
q2_hnsw_recall, q2_hnsw_hits, q2_hnsw_k = recall_stats(q2_hnsw_ids, gt_filtered, k_default)

rows.append({
    "mode": "hnsw_default",
    "latency_ms": round(q2_hnsw_latency_ms, 2),
    "recall": round(q2_hnsw_recall, 3),
    "returned_rows": len(q2_hnsw_df),
    "expected_k": k_default,
})

pd.DataFrame(rows)

,mode,latency_ms,recall,returned_rows,expected_k
0,no_index,4.01,1.0,10,10
1,hnsw_default,0.83,0.1,1,10


## 6) Oversampling experiment (`k` = 20, 30, 40)

Keep HNSW enabled and increase `k`.

- Observe how result size as `k` grows.

- Observe whether returned row count matches requested `k` and if they differ explain why ? 

In [10]:
oversampling_k_values = [20, 30, 40]
rows = []

con.execute("DROP INDEX IF EXISTS docs_hnsw_idx")
con.execute("SET hnsw_ef_search = 64")
con.execute(
    "CREATE INDEX docs_hnsw_idx ON docs USING HNSW (embedding) "
    "WITH (M = 4, ef_construction = 16)"
)

for k in oversampling_k_values:
    sql = Q2_SQL_TEMPLATE.format(k=k)
    df, _, _ = run_query_median(con, sql, runs=5)

    rows.append({
        "k": k,
        "returned_rows": len(df),
    })

pd.DataFrame(rows)

,k,returned_rows
0,20,2
1,30,5
2,40,5
